In [1]:
os.environ["SAGE_NUM_THREADS"] = '10'
# import logging
# logging.basicConfig()
# logging.getLogger('lefschetz_family.integrator').setLevel(logging.INFO)
# logging.getLogger('lefschetz_family.hypersurface').setLevel(logging.INFO)
# logging.getLogger('lefschetz_family.integrator_simultaneous').setLevel(logging.INFO)

In [2]:
from lefschetz_family import EllipticSurface

These are the values of the Bs at which we do the computations

In [3]:
values = [3,6,5,7]

In [4]:
R.<X,Y,Z> = QQ[]
S.<t> = R[]
T.<u> = S[]
B1, B2, B3, B4 = 3+u, 6-u, 5+2*u, 7 +3*u
basepoint = 0
values = [m(basepoint) for m in [B1, B2, B3, B4]]

B12=X
B23=Y
B13 = (B1+B2+B3+B4)*Z - B12 - B23
s2_12 = B1*B2 + B3*B4
s2_23 = B2*B3 + B1*B4
s2_13 = B1*B3 + B2*B4
s3 = B1*B2*B3 + B1*B2*B4 + B1*B3*B4 + B2*B3*B4

P = -81*t*Z^3 + B12*B23*B13 - s2_12*B12*Z^2 - s2_23*B23*Z^2 - s2_13*B13*Z^2 + 2*s3*Z^3
P

-5*Z^3*u^3 + (-6*X*Z^2 - 2*Y*Z^2 + 25*Z^3)*u^2 + (5*X*Y*Z - 10*X*Z^2 - Y*Z^2 + 27*Z^3)*u - 81*Z^3*t - X^2*Y - X*Y^2 + 21*X*Y*Z + 4*X*Z^2 + 6*Y*Z^2 - 135*Z^3

That is the line we choose for monodromy. 

In [5]:
B1, B2, B3, B4

(u + 3, -u + 6, 2*u + 5, 3*u + 7)

In [6]:
forms = []
newforms = [1/P(t=t^4)]
for i in range(3):
    newforms += [newforms[-1].derivative()]
for i in range(4):
    newforms[i] *= P(t=t^4)^(i+1)
    newforms[i] = S(newforms[i](basepoint))
forms += newforms

forms += [t^i for i in range(1,11)]
forms += [t^i*X^3 for i in range(1,11)]

forms = [forms[i] for i in [0,1,2,3,4,5,6,9,10,12,13,14,15,18]]

len(forms)

14

In [7]:
# logging.getLogger('lefschetz_family.integrator').setLevel(logging.WARNING)

In [8]:
rat = EllipticSurface(P(basepoint), nbits=1000, fibration = [vector(ZZ, [6, 1, -3]), vector(ZZ, [6, 5, 0])])
print("Rational surface")
print("MW:", 10-matrix(rat.trivial_lattice).rank())
print("types:", rat.types)

Rational surface
MW: 2
types: ['I1', 'I1', 'I1', 'I1', 'IV*']


We start by computing the periods and recovering some invariants of the surface: it has one $IV^*$ fibre, its Néron-Séveri is $\mathbb Z/6\mathbb Z$, its Picard rank is 14 (and therefore the parabolic homology has rank $22-14+6 =14$), and its discriminant is $2^6$.

In [9]:
K3 = EllipticSurface(P(basepoint)(t=t^4), nbits=1000, simultaneous_integration=False, fibration=[vector(ZZ, [8, 1, -1]), vector(ZZ, [-4, 9, -2])])
print("K3 surface")
print("types:", K3.types)
print("MW:", K3.mordell_weil)
print("Picard rank:", len(K3.neron_severi))
print("discriminant:", (matrix(K3.neron_severi)*K3.intersection_product*matrix(K3.neron_severi).transpose()).det())

K3 surface
types: ['I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'IV*']
MW: Finitely generated module V/W over Integer Ring with invariants (0, 0, 0, 0, 0, 0)
Picard rank: 14
discriminant: -64


Furthermore the Picard lattice is the only 2-elementary lattice with invariants $r=14=1+13$, $l=6$, and $\delta=0$, that is $U\oplus D_4\oplus D_4\oplus D_4$

In [10]:
IntegralLattice(matrix(K3.neron_severi) * K3.intersection_product * matrix(K3.neron_severi).transpose()).discriminant_group()

Finite quadratic module over Integer Ring with invariants (2, 2, 2, 2, 2, 2)
Gram matrix of the quadratic form with values in Q/2Z:
[  1 1/2   0 1/2   0 1/2]
[1/2   1   0   0   0   0]
[  0   0   1   0   0 1/2]
[1/2   0   0   1 1/2   0]
[  0   0   0 1/2   1 1/2]
[1/2   0 1/2   0 1/2   1]

Numerical approximation of the period matrix on $H_2(X)$

In [11]:
%time permat = K3.integrate_forms(forms)

CPU times: user 2.74 s, sys: 9.83 s, total: 12.6 s
Wall time: 2min 56s


`sigma` is the action of the order 4 automorphism on the parabolic homology

We compute the period matrix on just parabolic homology `permat_red`. This allows us to compute the action of the $\mathbb Z/4\mathbb Z$ automorphism on the parabolic homology.

In [12]:
parabolic_lattice = (K3.intersection_product * matrix(K3.trivial_lattice).transpose()).left_kernel_matrix()
permat_red = (permat * parabolic_lattice.transpose())
sigma = (permat_red.inverse() * I * diagonal_matrix([1,1,1,1,I,I^2,I^3,I^6,I^7,I^9,I^10,I,I^2,I^5]) * permat_red).change_ring(ZZ)
sigma

[-1  0  0  0 -1  0  0  0 -1  0  0  0  0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -1 -1 -1]
[ 1  0  0  0  1  0  0  0  1  0  0  0  0  1]
[ 1  0  0  0  1  0  0  0  1  0  0  0  1  0]
[ 1  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 0  0  0  1  0  0  0  0  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0  0  0  0  0  0  0]
[ 0  0  0  0  1  0  0  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  1  0  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  1  0  0  1  0  0]
[ 0  0  0  0  0  0  0  0  1  0  1  0  0  0]

From this we certify the transcendental lattice (equivalently $L_-$ in the notation of Kondo) computation (as the kernel of $\sigma^2+1$). We do a change of basis to the make explicit the shape $U(2)\oplus U(2)\oplus D_4$

In [13]:
IntegralLattice("U").direct_sum(IntegralLattice("U")).twist(2).direct_sum(IntegralLattice("D4")).discriminant_group()

Finite quadratic module over Integer Ring with invariants (2, 2, 2, 2, 2, 2)
Gram matrix of the quadratic form with values in Q/2Z:
[  0 1/2   0   0   0   0]
[1/2   0   0   0   0   0]
[  0   0   1 1/2 1/2   0]
[  0   0 1/2   0   0   0]
[  0   0 1/2   0   1 1/2]
[  0   0   0   0 1/2   1]

In [14]:
IntegralLattice(matrix(K3.transcendental_lattice)*K3.intersection_product*matrix(K3.transcendental_lattice).transpose()).discriminant_group()

Finite quadratic module over Integer Ring with invariants (2, 2, 2, 2, 2, 2)
Gram matrix of the quadratic form with values in Q/2Z:
[  1 1/2   0 1/2 1/2 1/2]
[1/2   1   0 1/2 1/2 1/2]
[  0   0   1 1/2 1/2 1/2]
[1/2 1/2 1/2   1 1/2 1/2]
[1/2 1/2 1/2 1/2   1 1/2]
[1/2 1/2 1/2 1/2 1/2   1]

In [15]:
TrX = (sigma^2+1).transpose().kernel().basis_matrix()
TrX = (TrX * parabolic_lattice)
CB = identity_matrix(8)
CB = Permutation([1,5,2,8,3,7,4,6]).to_matrix().transpose() * CB
TrX = CB * TrX 
CBtemp = matrix(ZZ, [[1, 1, 0, -2, 0, -1, 0, -1], [1, 1, 1, -1, 0, -1, 0, -1], [-2, 0, 0, 2, 1, 1, 1, 1], [-1, 1, 1, 0, 0, 0, 1, 0], [-1, 0, 0, 1, 0, 1, 1, 1], [1, 0, 0, -1, 0, 0, -1, 0], [1, 0, 0, -1, 0, 0, 0, -1], [-1, 0, 0, 1, 0, 0, 0, 0]])
TrX = CBtemp * TrX
TrX*K3.intersection_product*TrX.transpose()

[ 0  2  0  0  0  0  0  0]
[ 2  0  0  0  0  0  0  0]
[ 0  0  0  2  0  0  0  0]
[ 0  0  2  0  0  0  0  0]
[ 0  0  0  0 -2  0  0  1]
[ 0  0  0  0  0 -2  0  1]
[ 0  0  0  0  0  0 -2  1]
[ 0  0  0  0  1  1  1 -2]

We further refine this basis so that $\sigma$ has a block diagonal shape

In [16]:
v1, v2 = parabolic_lattice.solve_left(TrX).rows()[7], parabolic_lattice.solve_left(TrX).rows()[5]
mat = matrix([v1, v2, sigma*v1, sigma*v2])
mat[2], mat[3] = mat[3], mat[2]
mat[0] += mat[3] + mat[1]
mat[0] *= -1
(mat * parabolic_lattice * K3.intersection_product * parabolic_lattice.transpose() * mat.transpose() )

Us = (TrX * K3.intersection_product * parabolic_lattice.transpose() * mat.transpose()).left_kernel_matrix() * parabolic_lattice.solve_left(TrX)
Us[0], Us[1], Us[2], Us[3] = Us[1], sigma*Us[1], -Us[3], -sigma*Us[3]

mat = matrix([mat[0], sigma*mat[0], mat[3], sigma*mat[3]])

TrX = Us.stack(mat) * parabolic_lattice
TrX * K3.intersection_product * TrX.transpose()

[ 0  0  0  2  0  0  0  0]
[ 0  0 -2  0  0  0  0  0]
[ 0 -2  0  0  0  0  0  0]
[ 2  0  0  0  0  0  0  0]
[ 0  0  0  0 -2  0  1 -1]
[ 0  0  0  0  0 -2  1  1]
[ 0  0  0  0  1  1 -2  0]
[ 0  0  0  0 -1  1  0 -2]

In [17]:
action_on_TrX = parabolic_lattice.solve_left(TrX).transpose().solve_right(sigma * parabolic_lattice.solve_left(TrX).transpose())
action_on_TrX

[ 0 -1  0  0  0  0  0  0]
[ 1  0  0  0  0  0  0  0]
[ 0  0  0 -1  0  0  0  0]
[ 0  0  1  0  0  0  0  0]
[ 0  0  0  0  0 -1  0  0]
[ 0  0  0  0  1  0  0  0]
[ 0  0  0  0  0  0  0 -1]
[ 0  0  0  0  0  0  1  0]

The $i$-eigenspace and its hermitian form is then easily obtainable

In [18]:
IEigenspace = matrix([TrX[i] - I*TrX[i+1] for i in [0,2,4,6]])
IEigenspace[0] *=-I
Hermitian_form = IEigenspace * K3.intersection_product * IEigenspace.conjugate().transpose()
Hermitian_form

[       0        4        0        0]
[       4        0        0        0]
[       0        0       -4 -2*I + 2]
[       0        0  2*I + 2       -4]

### Monodromy computation

We start by computing the Picard-Fuchs equation that we will use to compute monodromy

In [19]:
from ore_algebra import OreAlgebra
from lefschetz_family.voronoi import FundamentalGroupVoronoi
from lefschetz_family.integrator import Integrator
from lefschetz_family.numperiods.integerRelations import IntegerRelations

In [20]:
try:
    mathematica('Needs["RISC`HolonomicFunctions`"]')
except:
    pass
mathematica('Needs["RISC`HolonomicFunctions`"]')

Null

In [21]:
%%time
Qu.<u> = QQ[]
Su.<Du> = OreAlgebra(Qu)
mathematica("f="+str(P(t=t^4)))
mathematica("R = 1/f")
mathematica("ann = Annihilator[R, {Der[X], Der[Y], Der[Z], Der[t], Der[u]}]")
mathematica("ann1 = CreativeTelescoping[ann , Der[X], { Der[Y], Der[Z], Der[t], Der[u] }][[1]]")
mathematica("ann2 = CreativeTelescoping[ann1, Der[Y], { Der[Z], Der[t], Der[u] }][[1]]")
mathematica("ann3 = CreativeTelescoping[ann2, Der[Z], { Der[t], Der[u] }][[1]]")
mathematica("annfin = CreativeTelescoping[ann3, Der[t], { Der[u] }][[1]]")
Llist = mathematica("annfin[[1]][[1]]").sage()
L = sum([Qu(str(c)) * Su.gen()**ZZ(p[0]) for c, p in Llist])

CPU times: user 9.09 ms, sys: 7.06 ms, total: 16.2 ms
Wall time: 13.2 s


In [22]:
L.order()

4

In [23]:
def derivative(P, A, v):
    k = ZZ((A(1)(1).degree()+3)/3)
    newnum = 1/k*P*A.derivative(v) - A*P.derivative(v)
    return newnum

In [24]:
ders = [P.parent()(1)]
for i in range(3):
    ders += [derivative(P(t=t^4), ders[-1], P.parent().gen())]
ders = [v(basepoint) for v in ders]

We compute a basis of the fundamental group and the action of monodromy on the Frobenius basis at the base point.

In [25]:
roots = L.leading_coefficient().roots(QQbar, multiplicities=False)
fg = FundamentalGroupVoronoi(roots, basepoint)
_ = fg.sort_loops()
integrator = Integrator(fg, L, 1200)
monodromy_on_frobenius = integrator.transition_matrices

In [26]:
def get_decomp(c, decomp, replace):
    if 0 in c:
        return 0
    res = IntegerRelations(matrix([c]+[CBF(u) for u in decomp]).transpose()).basis.row(0)
    return - res[1:] * vector(replace) / res[0]

We compute the periods at the base point, which will serve as a base change between the Frobenius basis and the homology basis.

In [27]:
%time pertemp = K3.integrate_forms(ders)

CPU times: user 844 ms, sys: 3.4 s, total: 4.24 s
Wall time: 54.7 s


In [28]:
period_on_IEigenspace = pertemp * IEigenspace.transpose()

In [29]:
integration_correction = diagonal_matrix([1/factorial(i) for i in range(4)])
homology_to_frobenius = integration_correction * period_on_IEigenspace

In [30]:
Ms = [(homology_to_frobenius.inverse() * tM * homology_to_frobenius).apply_map(lambda c: get_decomp(c, [1,I],[1,I])) for tM in monodromy_on_frobenius]

As a sanity check, we can verify that they respect the Hermitian form

In [31]:
[M.transpose() * Hermitian_form* M.conjugate() == Hermitian_form for M in Ms]

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True]

In [32]:
for M, critical_value, loop in zip(Ms, fg.points[1:], fg.pointed_loops_vertices):
    if M==1:
        continue
    print("critical value:",critical_value)
    print("minimal polynomial of critical value:",critical_value.minpoly())
    print("value of Bs at critical value:", [m(critical_value) for m in [B1, B2, B3, B4]])
    print("loop:", loop.path)
    print(M)
    print("-"*60)
    print("\n"*3)

critical value: -0.7055665131205157?
minimal polynomial of critical value: x^4 - 276/409*x^3 - 2666/409*x^2 - 340/409*x + 889/409
value of Bs at critical value: [2.294433486879485?, 6.705566513120516?, 3.588866973758969?, 4.883300460638453?]
loop: [0, -33/19*I - 6/17, 33/19*I - 6/17, 31/21*I - 32/53, -27/22, -31/21*I - 32/53, -33/19*I - 6/17, 0]
[     1      2  I - 1 -I + 1]
[     0      1      0      0]
[     0      2      I -I + 1]
[     0   -2*I  I + 1     -I]
------------------------------------------------------------




critical value: -2.052389311019607?
minimal polynomial of critical value: x^4 - 276/409*x^3 - 2666/409*x^2 - 340/409*x + 889/409
value of Bs at critical value: [0.9476106889803931?, 8.05238931101961?, 0.8952213779607861?, 0.842832066941180?]
loop: [0, -33/19*I - 6/17, -31/21*I - 32/53, -13/15*I - 9/5, -4/15*I - 17/9, -177/106, 4/15*I - 17/9, 5/24*I - 57/26, -5/24*I - 57/26, -4/15*I - 17/9, -13/15*I - 9/5, -31/21*I - 32/53, -33/19*I - 6/17, 0]
[       3        2  

### We express everything embedded in the full K3 lattice recovered by `lefschetz-family`

In [33]:
show(K3.intersection_product)

22 x 22 dense matrix over Integer Ring (use the '.str()' method to see the entries)

In [34]:
parabolic_lattice

[ 1  0  0  0  0  0  0  0  0  0  0  0  0  0 -3  1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  1  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  1  0  0  0  0  0  0  0  0  0 -3  1  0  0  0  0  0  0]
[ 0  0  0  0  0  1  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  1  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  1  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  1  0  0  0  0  0 -3  1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  1  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  1  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  1  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  1  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  0]

In [35]:
TrX

[ 1  1  0  0  1 -1 -1 -1 -1 -1  0  0  0  0  0  1  0  0  0  0  0  0]
[-1  1  1  1  1  0  0  1  1 -1 -1 -1 -1 -1 -2  1  0  0  0  0  0  0]
[ 1 -1  0 -1 -1  0  0  0 -1  1  0  1  1  1  1 -1  0  0  0  0  0  0]
[ 1  0  0  0  1 -1  0 -1 -1  0  0  0  0 -1  0  1  0  0  0  0  0  0]
[ 0  1  0  1  1  0  0  0  0 -1  0 -1 -1 -1 -1  1  0  0  0  0  0  0]
[-1  0  0  0  0  1  0  1  1  0  0  0 -1  0 -1  0  0  0  0  0  0  0]
[ 0 -1  0  0 -1  0  0  0  0  1  0  0  1  1  1 -1  0  0  0  0  0  0]
[ 1  0  0  0  0  0  0 -1 -1  0  0  0  0  0  1  0  0  0  0  0  0  0]

In [36]:
parabolic_lattice.solve_left(TrX) # The coordinates of TrX in the basis of the parabolic lattice

[ 1  1  0  0  1 -1 -1 -1 -1 -1  0  0  0  0]
[-1  1  1  1  1  0  0  1  1 -1 -1 -1 -1 -1]
[ 1 -1  0 -1 -1  0  0  0 -1  1  0  1  1  1]
[ 1  0  0  0  1 -1  0 -1 -1  0  0  0  0 -1]
[ 0  1  0  1  1  0  0  0  0 -1  0 -1 -1 -1]
[-1  0  0  0  0  1  0  1  1  0  0  0 -1  0]
[ 0 -1  0  0 -1  0  0  0  0  1  0  0  1  1]
[ 1  0  0  0  0  0  0 -1 -1  0  0  0  0  0]

In [37]:
TrX * K3.intersection_product * TrX.transpose() # the intersection product in this basis of TrX

[ 0  0  0  2  0  0  0  0]
[ 0  0 -2  0  0  0  0  0]
[ 0 -2  0  0  0  0  0  0]
[ 2  0  0  0  0  0  0  0]
[ 0  0  0  0 -2  0  1 -1]
[ 0  0  0  0  0 -2  1  1]
[ 0  0  0  0  1  1 -2  0]
[ 0  0  0  0 -1  1  0 -2]

In [38]:
show(IEigenspace)

[-I + 1 -I - 1     -1     -1 -I - 1      I      I  I - 1  I - 1  I + 1      1      1      1      1      2 -I - 1      0      0      0      0      0      0]
[-I + 1     -1      0     -1 -I - 1      I      0      I  I - 1      1      0      1      1  I + 1      1 -I - 1      0      0      0      0      0      0]
[     I      1      0      1      1     -I      0     -I     -I     -1      0     -1  I - 1     -1  I - 1      1      0      0      0      0      0      0]
[    -I     -1      0      0     -1      0      0      I      I      1      0      0      1      1 -I + 1     -1      0      0      0      0      0      0]

In [39]:
TrX.solve_left(IEigenspace) # coordinates of the I eigenspace in the basis of TrX

[-I -1  0  0  0  0  0  0]
[ 0  0  1 -I  0  0  0  0]
[ 0  0  0  0  1 -I  0  0]
[ 0  0  0  0  0  0  1 -I]

### The discriminant group

In [40]:
generators_of_discr = matrix([g.lift() for g in IntegralLattice(TrX * K3.intersection_product * TrX.transpose()).discriminant_group().gens()])
generators_of_discr

[   0    0  1/2    0    0    0    0    0]
[   0    0    0  1/2    0    0    0    0]
[   0  1/2    0    0 -1/2  1/2    0    0]
[   0    0    0    0 -1/2  1/2    0    0]
[ 1/2    0    0    0    0    0 -1/2  1/2]
[   0    0    0    0    0    0 -1/2  1/2]

In [41]:
generators_of_discr * TrX

[ 1/2 -1/2    0 -1/2 -1/2    0    0    0 -1/2  1/2    0  1/2  1/2  1/2  1/2 -1/2    0    0    0    0    0    0]
[ 1/2    0    0    0  1/2 -1/2    0 -1/2 -1/2    0    0    0    0 -1/2    0  1/2    0    0    0    0    0    0]
[  -1    0  1/2    0    0  1/2    0    1    1    0 -1/2    0 -1/2    0   -1    0    0    0    0    0    0    0]
[-1/2 -1/2    0 -1/2 -1/2  1/2    0  1/2  1/2  1/2    0  1/2    0  1/2    0 -1/2    0    0    0    0    0    0]
[   1    1    0    0    1 -1/2 -1/2   -1   -1   -1    0    0 -1/2 -1/2    0    1    0    0    0    0    0    0]
[ 1/2  1/2    0    0  1/2    0    0 -1/2 -1/2 -1/2    0    0 -1/2 -1/2    0  1/2    0    0    0    0    0    0]

### Fibre types on discriminant

In [42]:
R.<X,Y,Z> = QQ[]
S.<t> = R[]
values = [0,6,5,7] # one B equal to zero
basepoint = 0
B1, B2, B3, B4 = values

B12=X
B23=Y
B13 = (B1+B2+B3+B4)*Z - B12 - B23
s2_12 = B1*B2 + B3*B4
s2_23 = B2*B3 + B1*B4
s2_13 = B1*B3 + B2*B4
s3 = B1*B2*B3 + B1*B2*B4 + B1*B3*B4 + B2*B3*B4

P = -81*t*Z^3 + B12*B23*B13 - s2_12*B12*Z^2 - s2_23*B23*Z^2 - s2_13*B13*Z^2 + 2*s3*Z^3
P.parent()

Univariate Polynomial Ring in t over Multivariate Polynomial Ring in X, Y, Z over Rational Field

In [43]:
K32 = EllipticSurface(P(t=t^4), nbits=1000, simultaneous_integration=False, fibration=[vector(ZZ, [8, 1, -1]), vector(ZZ, [-4, 9, -2])])
print("K3 surface")
print("types:", K32.types)
print("MW:", K32.mordell_weil)
print("Picard rank:", len(K32.neron_severi))
print("discriminant:", (matrix(K32.neron_severi)*K32.intersection_product*matrix(K32.neron_severi).transpose()).det())

K3 surface
types: ['I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I4', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'IV*']
MW: Finitely generated module V/W over Integer Ring with invariants (0, 0, 0, 0, 0)
Picard rank: 16
discriminant: -64


In [44]:
R.<X,Y,Z> = QQ[]
S.<t> = R[]
values = [2^2,3^2,1^2,6^2] # sum of As equal to zero
basepoint = 0
B1, B2, B3, B4 = values

B12=X
B23=Y
B13 = (B1+B2+B3+B4)*Z - B12 - B23
s2_12 = B1*B2 + B3*B4
s2_23 = B2*B3 + B1*B4
s2_13 = B1*B3 + B2*B4
s3 = B1*B2*B3 + B1*B2*B4 + B1*B3*B4 + B2*B3*B4

P = -81*t*Z^3 + B12*B23*B13 - s2_12*B12*Z^2 - s2_23*B23*Z^2 - s2_13*B13*Z^2 + 2*s3*Z^3
P

-81*Z^3*t - X^2*Y - X*Y^2 + 50*X*Y*Z + 256*X*Z^2 + 175*Y*Z^2 - 12800*Z^3

In [45]:
K32 = EllipticSurface(P(t=t^4), nbits=1000, simultaneous_integration=False, fibration=[vector(ZZ, [8, 1, -1]), vector(ZZ, [-4, 9, -2])])
print("K3 surface")
print("types:", K32.types)
print("MW:", K32.mordell_weil)
print("Picard rank:", len(K32.neron_severi))
print("discriminant:", (matrix(K32.neron_severi)*K32.intersection_product*matrix(K32.neron_severi).transpose()).det())

K3 surface
types: ['I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I4', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'IV*']
MW: Finitely generated module V/W over Integer Ring with invariants (0, 0, 0, 0, 0)
Picard rank: 16
discriminant: -64


### The Néron-Severi

In [46]:
matrix(K3.neron_severi)

[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1]
[ 0 -1  0  1  0  0  0  0  0 -1  0  1  0  0  0  0  0  0  0  0  0  0]
[ 0 -1  0  0  0  1  0  0  0 -1  0  0  1  0  0  0  0  0  0  0  0  0]
[ 0 -1  1  0  0  0  0  0  0 -1  1  0  0  0  0  0  0  0  0  0  0  0]
[ 0  0 -1  0  0  0  1  0  0  0 -1  0  0  1  0  0  0  0  0  0  0  0]
[ 0  0  0 -1  0  0  0  1  0  0  0 -1  0  0  1  0  0  0  0  0  0  0]
[ 1  0  0 -1  0  0  0  0  0  0  0 -1  0  0  0  1  0  0  0  0  0  0]
[ 0  0  0  0 -1  0  1  0  1  0  0  0 -1  0 -1  0  0  0  0  0  0  0]
[ 1  0  0 -1 -1  0  0  0  0  1  1  0  0  0  0 -1  0  0  0  0  0  0]

In [47]:
matrix(K3.neron_severi) * K3.intersection_product * matrix(K3.neron_severi).transpose()

[-2 -1 -1 -1  0  0  0  0  0  0  0 -1  1  0]
[-1 -2  0 -1  0  0  0  0  0  0  0 -1  1  1]
[-1  0 -2 -1  0  0  0  0  0  0  0 -1  1  0]
[-1 -1 -1 -2  0  0  0  0  0  0  0 -1  1  1]
[ 0  0  0  0  0  1  0  0  0  0  0  0  0  0]
[ 0  0  0  0  1 -2  0  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0 -4 -2 -2  0  2  2  0  2]
[ 0  0  0  0  0  0 -2 -4 -2  0  0  0  2  2]
[ 0  0  0  0  0  0 -2 -2 -4  2  0  0  0  0]
[ 0  0  0  0  0  0  0  0  2 -4  0  0  0  2]
[ 0  0  0  0  0  0  2  0  0  0 -4 -2  2  0]
[-1 -1 -1 -1  0  0  2  0  0  0 -2 -4  2  0]
[ 1  1  1  1  0  0  0  2  0  0  2  2 -4 -2]
[ 0  1  0  1  0  0  2  2  0  2  0  0 -2 -4]

In [48]:
matrix([K3.lift(v) for v in K3.components_of_singular_fibres[-1]]) # components of the IV* fibre

[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0]
[-1  1  1  1  2 -1 -1 -1 -1  0  0  0  0  0  0  1  1 -1  0 -1  0  0]
[-2  1  1  1  1  0  0  0  1 -1 -1 -1 -1 -1 -1  2  1  0  1 -1  0  0]

In [49]:
CBTrX_to_eigenspaces = TrX.solve_left(IEigenspace).stack(TrX.solve_left(IEigenspace).conjugate())

In [50]:
TrXlat = IntegralLattice(TrX * K3.intersection_product * TrX.transpose())

Checking that monodromy preserves the intersection product on the transcendental lattice

In [51]:
monodromy_on_TrX = [CBTrX_to_eigenspaces.transpose() * block_diagonal_matrix([M, M.conjugate()]) * CBTrX_to_eigenspaces.transpose().inverse() for M in Ms]
print([M.transpose() * TrXlat.gram_matrix() * M == TrXlat.gram_matrix() for M in monodromy_on_TrX])

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]


In [52]:
[M for M in monodromy_on_TrX if M!=1]

[
[ 1  0  0  2  1 -1 -1  1]  [ 3  0  0  2  1 -3  0  2]
[ 0  1 -2  0  1  1 -1 -1]  [ 0  3 -2  0  3  1 -2  0]
[ 0  0  1  0  0  0  0  0]  [ 0 -2  3  0 -3 -1  2  0]
[ 0  0  0  1  0  0  0  0]  [ 2  0  0  3  1 -3  0  2]
[ 0  0  2  0  0 -1  1  1]  [ 0 -4  4  0 -5 -2  4  0]
[ 0  0  0  2  1  0 -1  1]  [ 4  0  0  4  2 -5  0  4]
[ 0  0  0  2  1 -1  0  1]  [ 2  0  0  2  1 -3  1  2]
[ 0  0 -2  0  1  1 -1  0], [ 0  2 -2  0  3  1 -2  1],

[ 1  0 -1  1  1 -1  0  1]  [ 2  1 -1  1  1 -1 -1  0]
[ 0  1 -1 -1  1  1 -1  0]  [-1  2 -1 -1  1  1  0 -1]
[ 0  0  1  0  0  0  0  0]  [ 1 -1  2  1 -1 -1  0  1]
[ 0  0  0  1  0  0  0  0]  [ 1  1 -1  2  1 -1 -1  0]
[ 0  0  1  1  0 -1  1  0]  [ 3 -1  1  3  0 -3 -1  2]
[ 0  0 -1  1  1  0  0  1]  [ 1  3 -3  1  3  0 -2 -1]
[ 0  0  0  0  0  0  1  0]  [ 2  2 -2  2  2 -2 -1  0]
[ 0  0  0  0  0  0  0  1], [-2  2 -2 -2  2  2  0 -1],

[ 1  0  0  0  0  0  0  0]  [ 1  0  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0]  [ 0  1  0  0  0  0  0  0]
[ 1 -1  1  0 -1 -1  1  0]  [ 0  0  1  0 

The monodromy action on the discriminant group is $(\mathbb Z/2\mathbb Z)^4$, generated by the monodromy around the $B_i=0$ divisors

In [53]:
monodromy_on_discr = []
for M in monodromy_on_TrX:
    if M==1:
        continue
    monodromy_on_discr += [matrix(IntegerModRing(2), [TrXlat.discriminant_group()(M*v.lift()) for v in TrXlat.discriminant_group().gens()])]
monodromy_on_discr

[
[1 0 0 0 0 0]  [1 0 0 0 0 0]  [1 0 1 0 1 1]  [0 1 1 0 1 1]
[0 1 0 0 0 0]  [0 1 0 0 0 0]  [0 1 1 0 1 1]  [1 0 1 0 1 1]
[0 0 1 0 0 0]  [0 0 1 0 0 0]  [0 0 1 0 0 0]  [1 1 0 0 1 1]
[0 0 0 1 0 0]  [0 0 0 1 0 0]  [0 0 0 1 0 0]  [0 0 0 1 0 0]
[0 0 0 0 1 0]  [0 0 0 0 1 0]  [0 0 1 0 0 1]  [0 0 0 0 1 0]
[0 0 0 0 0 1], [0 0 0 0 0 1], [0 0 1 0 1 0], [1 1 1 0 1 0],

[1 0 0 0 0 0]  [1 0 0 0 0 0]  [1 0 0 0 0 0]  [1 0 0 0 0 0]
[0 1 0 0 0 0]  [0 1 0 0 0 0]  [0 1 0 0 0 0]  [0 1 0 0 0 0]
[1 1 1 1 0 0]  [0 0 1 0 0 0]  [0 0 1 0 0 0]  [0 0 1 0 0 0]
[0 0 0 1 0 0]  [0 0 0 1 0 0]  [0 0 0 1 0 0]  [0 0 0 1 0 0]
[0 0 0 0 1 0]  [0 0 0 1 1 0]  [0 0 0 0 1 0]  [0 0 0 0 1 0]
[1 1 0 1 0 1], [0 0 0 1 0 1], [0 0 0 0 0 1], [0 0 0 0 0 1]
]

In [54]:
MatrixGroup(monodromy_on_discr).group_id()

[16, 14]

### Computing the action of coordinate permutations on the transcendental lattice

In [55]:
R.<X,Y,Z> = QQ[]
S.<t> = R[]
T.<u> = S[]

In [56]:
def swap_action(B1, B2, B3, B4):
    basepoint = 0
    values = [m(basepoint) for m in [B1, B2, B3, B4]]
    
    B12, B23 = X, Y
    B13 = (B1+B2+B3+B4)*Z - B12 - B23
    s2_12 = B1*B2 + B3*B4
    s2_23 = B2*B3 + B1*B4
    s2_13 = B1*B3 + B2*B4
    s3 = B1*B2*B3 + B1*B2*B4 + B1*B3*B4 + B2*B3*B4
    P = -81*t*Z^3 + B12*B23*B13 - s2_12*B12*Z^2 - s2_23*B23*Z^2 - s2_13*B13*Z^2 + 2*s3*Z^3
    P
    
    Qu.<u> = QQ[]
    Su.<Du> = OreAlgebra(Qu)
    mathematica("f="+str(P(t=t^4)))
    mathematica("R = 1/f")
    mathematica("ann = Annihilator[R, {Der[X], Der[Y], Der[Z], Der[t], Der[u]}]")
    mathematica("ann1 = CreativeTelescoping[ann , Der[X], { Der[Y], Der[Z], Der[t], Der[u] }][[1]]")
    mathematica("ann2 = CreativeTelescoping[ann1, Der[Y], { Der[Z], Der[t], Der[u] }][[1]]")
    mathematica("ann3 = CreativeTelescoping[ann2, Der[Z], { Der[t], Der[u] }][[1]]")
    mathematica("annfin = CreativeTelescoping[ann3, Der[t], { Der[u] }][[1]]")
    Llist = mathematica("annfin[[1]][[1]]").sage()
    L = sum([Qu(str(c)) * Su.gen()**ZZ(p[0]) for c, p in Llist])
    
    ders = [P.parent()(1)]
    for i in range(3):
        ders += [derivative(P(t=t^4), ders[-1], P.parent().gen())]
    ders = [v(basepoint) for v in ders]
    roots = L.leading_coefficient().roots(QQbar, multiplicities=False) + [1]
    fg = FundamentalGroupVoronoi(roots, basepoint)
    _ = fg.sort_loops()
    path = [fg.vertices[v] for v in fg.paths[fg.points.index(1)-1]] + [1]
    swap = L.numerical_transition_matrix(path, eps=2^-1200)
    pertemp = K3.integrate_forms(ders)
    pertemp2 = diagonal_matrix([1,-1,1,-1]) * pertemp
    period_on_IEigenspace = pertemp * IEigenspace.transpose()
    period_on_IEigenspace2 = pertemp2 * IEigenspace.transpose()
    
    integration_correction = diagonal_matrix([1/factorial(i) for i in range(4)])
    homology_to_frobenius = integration_correction * period_on_IEigenspace
    homology_to_frobenius2 = integration_correction * period_on_IEigenspace2
    swap = homology_to_frobenius2.inverse() * swap * homology_to_frobenius
    swap = swap.apply_map(lambda c: get_decomp(c, [1,I],[1,I]))
    return swap

In [57]:
%%time
B1, B2, B3, B4 = 3+3*u, 6-3*u, T(5), T(7)
swapB1B2 = swap_action(B1,B2,B3,B4)

CPU times: user 1.67 s, sys: 4 s, total: 5.67 s
Wall time: 1min 4s


In [58]:
%%time
B1, B2, B3, B4 = 3+2*u, T(6), T(5-2*u), T(7)
swapB1B3 = swap_action(B1,B2,B3,B4)

CPU times: user 1.77 s, sys: 3.62 s, total: 5.39 s
Wall time: 1min 2s


In [59]:
%%time
B1, B2, B3, B4 = 3+4*u, T(6), T(5), T(7-4*u)
swapB1B4 = swap_action(B1,B2,B3,B4)

CPU times: user 1.91 s, sys: 4.27 s, total: 6.18 s
Wall time: 1min 7s


These matrices are defined only up to monodromy

In [60]:
swaps = [swapB1B2, swapB1B3, swapB1B4]
show(swaps)

[
[     1      0      0      0]  [     1      1      0     -I]
[     1      1      I -I + 1]  [     0      1      0      0]
[     2      0      I -I + 1]  [     0  I + 1     -I      0]
[-I + 1      0      0     -I], [     0      0 -I - 1      I],

[     0      1      0      0]
[     1      2  I - 1 -I + 1]
[     0      2      I -I + 1]
[     0   -2*I  I + 1     -I]
]

In [61]:
swaps_on_TrX = [CBTrX_to_eigenspaces.transpose() * block_diagonal_matrix([swap, swap.conjugate()]) * CBTrX_to_eigenspaces.transpose().inverse() for swap in swaps]

In [62]:
swaps_on_discr = [matrix(IntegerModRing(2), [TrXlat.discriminant_group()(swap*v.lift()) for v in TrXlat.discriminant_group().gens()]) for swap in swaps_on_TrX]

The group action on the discriminant induced by monodromy and permutation of the $B_i$'s is $W(B_4)\simeq (\mathbb Z/2\mathbb Z)^4\rtimes S_4$.

In [63]:
MatrixGroup(monodromy_on_discr + swaps_on_discr).group_id()

[384, 5602]

Adding triality, we obtain $W(F_4)$

In [64]:
trialityMatrix = matrix(4, [1,0,0,0,0,1,0,0,0,0,0,-1,0,0,1,-1])
trialityMatrix = CBTrX_to_eigenspaces.transpose() * block_diagonal_matrix([trialityMatrix, trialityMatrix.conjugate()]) * CBTrX_to_eigenspaces.transpose().inverse()
trialityMatrix = matrix(IntegerModRing(2), [TrXlat.discriminant_group()(trialityMatrix*v.lift()) for v in TrXlat.discriminant_group().gens()])

In [65]:
MatrixGroup(monodromy_on_discr + swaps_on_discr + [trialityMatrix]).group_id()

[1152, 157478]

And the full orthogonal group of the discriminant is $W(E_6)$

In [66]:
TrXlat.discriminant_group().orthogonal_group().cardinality()

51840

In [67]:
WG = WeylGroup("E6")

In [68]:
WG.is_isomorphic(TrXlat.discriminant_group().orthogonal_group())

True